In [1]:
from flask import Flask, request, render_template_string
import joblib

app = Flask(__name__)
model = joblib.load("fake_news_model.joblib")

PAGE = """
<!doctype html>
<html>
<head>
  <title>Fake News Detector</title>
  <style>
    body { font-family: Arial; background:#f4f6f8; margin:0; padding:40px; }
    .card { max-width:700px; margin:auto; background:white; padding:30px;
            border-radius:12px; box-shadow:0 4px 12px rgba(0,0,0,.1); }
    input, textarea { width:100%; padding:10px; margin:8px 0 18px;
                      border:1px solid #ccc; border-radius:6px; }
    button { background:#1a73e8; color:white; padding:12px 20px;
             border:none; border-radius:6px; cursor:pointer; }
    .result { margin-top:20px; padding:15px; border-radius:8px; }
    .fake { background:#ffe5e5; color:#b00020; }
    .real { background:#e6f7ea; color:#0b6623; }
  </style>
</head>
<body>
  <div class="card">
    <h1>Fake News Detector Using AI</h1>
    <form method="POST">
      <label>News Title</label>
      <input name="title" required>
      <label>News Text</label>
      <textarea name="text" rows="7" required></textarea>
      <button type="submit">Check News</button>
    </form>

    {% if result %}
      <div class="result {{ 'fake' if result.prediction == 'FAKE NEWS' else 'real' }}">
        <h3>{{ result.prediction }}</h3>
        <p>Confidence: {{ result.confidence }}%</p>
        <p>Fake probability: {{ result.fake_probability }}%</p>
        <p>Real probability: {{ result.real_probability }}%</p>
      </div>
    {% endif %}
  </div>
</body>
</html>
"""

@app.route("/", methods=["GET", "POST"])
def home():
    result = None
    if request.method == "POST":
        title = request.form.get("title", "")
        text = request.form.get("text", "")
        combined = f"{title} {text}"
        prediction = model.predict([combined])[0]
        probability = model.predict_proba([combined])[0]
        result = {
            "prediction": "FAKE NEWS" if prediction == 1 else "REAL NEWS",
            "confidence": round(probability[prediction] * 100, 2),
            "fake_probability": round(probability[1] * 100, 2),
            "real_probability": round(probability[0] * 100, 2)
        }
    return render_template_string(PAGE, result=result)

if __name__ == "__main__":
    app.run(debug=True)

ModuleNotFoundError: No module named 'joblib'